# Closira AI Agent Workflow

Four-stage notebook implementation for the assignment:

1. FAQ answering from the SOP only
2. Lead qualification
3. Continuous escalation detection
4. Structured conversation summary

Important: the `SOP_CONTENT` block below is intentionally preserved from the original notebook.


In [ ]:
!pip install google-genai


In [ ]:
import os
import re
from copy import deepcopy
from typing import Any, Dict, List, Optional

from google import genai
from google.genai import types

try:
    from google.colab import userdata
except Exception:
    userdata = None


def get_secret(name: str) -> Optional[str]:
    """Read an API key from Colab userdata first, then environment variables."""
    if userdata is not None:
        try:
            value = userdata.get(name)
            if value:
                return value
        except Exception:
            pass
    return os.getenv(name)


GEMINI_API_KEY = (
    get_secret("Gemini-API")
    or get_secret("Gemini-Questionnaire")
    or get_secret("GEMINI_API_KEY")
)

MODEL_NAME = "gemini-2.5-flash"


def get_gemini_client():
    if not GEMINI_API_KEY:
        raise RuntimeError(
            "Gemini API key not found. Set Colab userdata 'Gemini-API' "
            "or environment variable GEMINI_API_KEY."
        )
    return genai.Client(api_key=GEMINI_API_KEY)


In [ ]:
SOP_CONTENT = """
STANDARD OPERATING PROCEDURE (SOP) - Bloom Aesthetics Clinic

CLINIC INFORMATION:
- Business Name: Bloom Aesthetics Clinic
- Address: XYZ Road, Delhi
- Operating Hours: Monday to Saturday, 9:00 am – 7:00 pm. Closed on Sundays and Bank Holidays.
- Contact Channels: WhatsApp: 123456789 | Website: www.samplebloomaesthetic.com | Email: sampleemail@bloomaesthetics.co.in
- Booking Method: Via WhatsApp or online booking form at www.samplebloomaesthetic.com/book. Walk-ins are not accepted.
- Cancellation Policy: Minimum 24 hours' notice required. Late cancellations or no-shows may incur a Rs. 300 rebooking fee.
- Consultations: Free, required for all new patients, 20–30 minutes, conducted in person.

SERVICES AND PRICING:
1. Anti-Wrinkle Injections (Botox)
   - Relaxes facial muscles to reduce fine lines and wrinkles (forehead, eyes, cheeks, lips)
   - Pricing: 1 area from Rs. 20000 | 2 areas from Rs. 28000 | 3 areas from Rs. 35000
   - Results: Visible in 3–5 days, full effect at 2 weeks, lasts 3–4 months

2. Dermal Fillers
   - Hyaluronic acid injections for volume, contour, and smoothing (lips, cheeks, nasolabial folds, jawline)
   - Pricing: Lips from Rs. 25000 | Cheeks from Rs. 30000 | Jawline from Rs. 35000
   - Results: Immediate, settles after 2 weeks, lasts 6–18 months

3. Skin Booster Injections
   - Micro-injections of hyaluronic acid for hydration and elasticity (face, neck, hands)
   - Pricing: From Rs. 18000 per session | Course of 3 sessions from Rs. 48000
   - Results: Gradual improvement over 4 weeks, optimal after 3 treatments

4. Initial Consultation
   - One-to-one assessment covering treatment suitability, medical history, and treatment planning
   - Pricing: Free of charge

OUT OF SCOPE (do not answer):
- Medical advice or clinical/health/safety questions about treatments
- Pricing negotiation
- Formal complaints
- Anything not covered in this SOP
"""

SYSTEM_PROMPT = f"""You are a customer service agent for Bloom Aesthetics Clinic.

STRICT RULES:
1. Answer ONLY using the SOP content provided below. No assumptions, no external knowledge.
2. If the answer is not found in the SOP, respond with exactly: "ESCALATE: I don't have information on that. Let me connect you with a human agent."
3. If the question is out of scope (medical advice, pricing negotiation, complaints), respond with exactly: "ESCALATE: This is outside what I can assist with. Let me connect you with a human agent."
4. Be concise, friendly, and professional.

SOP CONTENT:
{SOP_CONTENT}"""

QUALIFICATION_FIELDS = [
    "Working Hours",
    "Team Size",
    "Services Offered + Price",
    "Booking Method",
    "Available Time Slots",
]

QUALIFICATION_QUESTIONS = {
    "Working Hours": "To help get you set up, what are your typical working hours?",
    "Team Size": "How many people are on your team?",
    "Services Offered + Price": "Which services do you offer, and what are their prices?",
    "Booking Method": "What is your preferred booking method?",
    "Available Time Slots": "What appointment time slots are available for customers?",
}

HANDOFF_MESSAGE = "I'm connecting you with a human agent. Please hold on."
HUMAN_UNAVAILABLE_MESSAGE = "Our team is currently unavailable. We will follow up shortly."


In [ ]:
def create_initial_state() -> Dict[str, Any]:
    return {
        "stage": "faq",
        "transcript": [],
        "qualification": {field: None for field in QUALIFICATION_FIELDS},
        "current_field_index": 0,
        "unanswered_count": 0,
        "answered_topics": [],
        "sop_gaps": [],
        "escalated": False,
        "escalation": {
            "trigger_type": None,
            "description": None,
            "last_customer_message": None,
            "stage": None,
        },
        "last_user_input": None,
        "last_agent_response": None,
        "lead_retry_counts": {field: 0 for field in QUALIFICATION_FIELDS},
    }


state = create_initial_state()


def add_transcript(role: str, message: str, conversation: Optional[Dict[str, Any]] = None) -> None:
    conversation = conversation or state
    conversation["transcript"].append({"role": role, "message": message})
    if role == "User":
        conversation["last_user_input"] = message
    elif role == "Agent":
        conversation["last_agent_response"] = message


def format_transcript(conversation: Optional[Dict[str, Any]] = None) -> str:
    conversation = conversation or state
    return "\n".join(
        f"{entry['role']}: {entry['message']}"
        for entry in conversation["transcript"]
    )


In [ ]:
def _short_topic(user_input: str) -> str:
    cleaned = re.sub(r"[^a-zA-Z0-9 ]+", "", user_input).strip()
    words = cleaned.split()[:8]
    return " ".join(words) if words else "General enquiry"


def _looks_like_empty_or_skip(text: Optional[str]) -> bool:
    if text is None:
        return True
    normalized = text.strip().lower()
    return normalized in {"", "skip", "no", "none", "not now", "later", "n/a", "na"}


def _next_missing_field(conversation: Dict[str, Any]) -> Optional[str]:
    for index, field in enumerate(QUALIFICATION_FIELDS):
        if conversation["qualification"].get(field) is None:
            conversation["current_field_index"] = index
            return field
    return None


def _confirmation_text(conversation: Dict[str, Any]) -> str:
    details = "\n".join(
        f"- {field}: {conversation['qualification'].get(field) or 'Not Provided'}"
        for field in QUALIFICATION_FIELDS
    )
    return f"Just to confirm, here's what I have noted:\n{details}"


def _build_summary_prompt(conversation: Dict[str, Any]) -> str:
    qualification = conversation["qualification"]
    transcript = format_transcript(conversation)
    escalation = conversation["escalation"] if conversation["escalated"] else "No escalation"
    return f"""Generate a concise structured conversation summary for a human operator.

Required sections:
- Customer Intent
- Key Details Collected
- Questions Answered
- SOP Gaps
- Escalation Details
- Recommended Actions

Qualification fields:
{qualification}

Questions answered:
{conversation['answered_topics']}

SOP gaps:
{conversation['sop_gaps']}

Escalation:
{escalation}

Transcript:
{transcript}
"""


In [ ]:
def faq_stage(user_input):
    """Answer an inbound FAQ question using the unchanged SOP only."""
    global state
    client = get_gemini_client()

    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=user_input,
        config=types.GenerateContentConfig(
            system_instruction=SYSTEM_PROMPT,
        ),
    )

    answer = (response.text or "").strip()
    topic = _short_topic(user_input)

    if answer.startswith("ESCALATE:"):
        state["unanswered_count"] += 1
        state["sop_gaps"].append(topic)
    else:
        state["answered_topics"].append(topic)

    return answer


def lead_qualification_stage(state):
    """Ask one qualification question at a time and store the latest user answer."""
    user_input = state.get("last_user_input")
    active_field = _next_missing_field(state)

    if active_field is None:
        state["stage"] = "summary"
        return _confirmation_text(state)

    if user_input is None:
        return QUALIFICATION_QUESTIONS[active_field]

    if _looks_like_empty_or_skip(user_input):
        state["lead_retry_counts"][active_field] += 1
        if state["lead_retry_counts"][active_field] >= 2:
            state["qualification"][active_field] = "Not Provided"
        else:
            return f"I need this to complete setup. {QUALIFICATION_QUESTIONS[active_field]}"
    else:
        state["qualification"][active_field] = user_input.strip()

    next_field = _next_missing_field(state)
    if next_field is None:
        state["stage"] = "summary"
        return _confirmation_text(state)

    return QUALIFICATION_QUESTIONS[next_field]


def escalation_check(user_input, response):
    """Continuously detect whether the conversation should be handed to a human."""
    global state
    user_text = (user_input or "").strip()
    response_text = (response or "").strip()
    lowered = user_text.lower()

    explicit_patterns = [
        "human", "agent", "representative", "manager", "real person",
        "speak to someone", "talk to someone", "call me",
    ]
    angry_patterns = [
        "angry", "furious", "ridiculous", "useless", "terrible", "awful",
        "stop repeating", "not helpful", "bad service", "waste of time",
    ]
    complaint_patterns = [
        "complaint", "complain", "refund", "bad experience", "unhappy",
        "disappointed", "staff was rude",
    ]
    medical_patterns = [
        "safe", "heart condition", "pregnant", "allergy", "infection",
        "side effect", "medical", "doctor", "medicine", "health", "risk",
    ]
    pricing_negotiation_patterns = [
        "discount", "negotiate", "reduce price", "cheaper", "price match",
        "lower the price", "best price",
    ]

    is_all_caps_frustration = (
        len(user_text) >= 12
        and any(char.isalpha() for char in user_text)
        and user_text.upper() == user_text
    )

    trigger_type = None
    description = None

    if response_text.startswith("ESCALATE:"):
        trigger_type = "Low Confidence / Out of Scope"
        description = response_text.replace("ESCALATE:", "", 1).strip()
    elif state.get("unanswered_count", 0) > 2:
        trigger_type = "Low Confidence"
        description = "More than two questions were unanswered from the SOP."
    elif any(pattern in lowered for pattern in explicit_patterns):
        trigger_type = "Explicit Request"
        description = "Customer asked to speak with a human agent."
    elif is_all_caps_frustration or any(pattern in lowered for pattern in angry_patterns):
        trigger_type = "Angry Sentiment"
        description = "Customer message shows frustration or angry sentiment."
    elif any(pattern in lowered for pattern in complaint_patterns):
        trigger_type = "Complaint"
        description = "Customer raised a complaint or negative service issue."
    elif any(pattern in lowered for pattern in medical_patterns):
        trigger_type = "Out of Scope"
        description = "Customer asked a medical, health, or safety question."
    elif any(pattern in lowered for pattern in pricing_negotiation_patterns):
        trigger_type = "Out of Scope"
        description = "Customer requested pricing negotiation."

    result = {
        "escalate": trigger_type is not None,
        "trigger_type": trigger_type,
        "description": description,
    }

    if result["escalate"] and not state.get("escalated"):
        state["escalated"] = True
        state["escalation"] = {
            "trigger_type": trigger_type,
            "description": description,
            "last_customer_message": user_input,
            "stage": state.get("stage"),
        }

    return result


def summary_stage(conversation):
    """Generate the required structured end-of-session summary."""
    try:
        client = get_gemini_client()
        response = client.models.generate_content(
            model=MODEL_NAME,
            contents=_build_summary_prompt(conversation),
            config=types.GenerateContentConfig(
                system_instruction=(
                    "You summarize customer support conversations for SMB operators. "
                    "Use only the provided conversation state and transcript. "
                    "Do not invent missing details; write Not Provided where needed."
                )
            ),
        )
        if response.text:
            return response.text.strip()
    except Exception:
        pass

    escalation = conversation["escalation"] if conversation["escalated"] else None
    details = "\n".join(
        f"- {field}: {conversation['qualification'].get(field) or 'Not Provided'}"
        for field in QUALIFICATION_FIELDS
    )
    return f"""--- Conversation Summary ---
Customer Intent: {conversation['answered_topics'][0] if conversation['answered_topics'] else 'Not Provided'}

Key Details Collected:
{details}

Questions Answered: {', '.join(conversation['answered_topics']) if conversation['answered_topics'] else 'None'}
SOP Gaps: {', '.join(conversation['sop_gaps']) if conversation['sop_gaps'] else 'None'}
Escalation Details: {escalation if escalation else 'No escalation'}
Recommended Actions: {'Human agent should review and follow up.' if escalation else 'Continue normal follow-up or onboarding.'}
"""


In [ ]:
def handle_escalation(user_input: str, response: str) -> bool:
    check = escalation_check(user_input, response)
    if not check["escalate"]:
        return False

    print("Agent:", HANDOFF_MESSAGE)
    add_transcript("Agent", HANDOFF_MESSAGE)
    print("\n", summary_stage(state))
    return True


def run_conversation():
    """Simple notebook/CLI runner for the full staged workflow."""
    global state
    state = create_initial_state()

    print("Agent: Hello! Ask me about Bloom Aesthetics Clinic, or type 'qualify' to start setup questions.")

    while True:
        user_input = input("User: ").strip()
        if user_input.lower() in {"exit", "quit", "end"}:
            break

        add_transcript("User", user_input)

        if user_input.lower() in {"qualify", "start qualification", "lead qualification"}:
            state["stage"] = "qualification"
            state["last_user_input"] = None
            response = lead_qualification_stage(state)
        elif state["stage"] == "qualification":
            response = lead_qualification_stage(state)
        else:
            response = faq_stage(user_input)

        add_transcript("Agent", response)
        print("Agent:", response)

        if handle_escalation(user_input, response):
            return state

        if state["stage"] == "summary":
            print("\n", summary_stage(state))
            return state

    print("\n", summary_stage(state))
    return state


def save_transcript_to_file(conversation=None, file_path="AI_agent_workflow_transcript.txt"):
    conversation = conversation or state
    with open(file_path, "w", encoding="utf-8") as f:
        f.write(summary_stage(conversation))
        f.write("\n\n--- Full Transcript ---\n\n")
        f.write(format_transcript(conversation))
    print(f"Transcript saved at: {file_path}")


## Optional Interactive Run

Run the next cell to chat with the workflow manually. Type `qualify` to move from FAQ handling into lead qualification.


In [ ]:
# Uncomment to run interactively in the notebook.
# final_state = run_conversation()
# save_transcript_to_file(final_state)


## Test Scenarios

These cells cover the PDF-required behaviours. FAQ calls require a valid Gemini API key; escalation and qualification checks are deterministic.


In [ ]:
# Test 1: In-SOP question
# Expected: answer accurately from SOP only.
# state = create_initial_state()
# q = "What are your Botox prices?"
# add_transcript("User", q)
# answer = faq_stage(q)
# add_transcript("Agent", answer)
# print(answer)
# print(escalation_check(q, answer))


In [ ]:
# Test 2: Out-of-scope question
# Expected: escalate instead of guessing.
# state = create_initial_state()
# q = "Do you offer dental implants?"
# add_transcript("User", q)
# answer = faq_stage(q)
# add_transcript("Agent", answer)
# print(answer)
# print(escalation_check(q, answer))


In [ ]:
# Test 3: Escalation trigger from angry sentiment / complaint
state = create_initial_state()
state["stage"] = "faq"
angry_message = "THIS IS RIDICULOUS AND I WANT A HUMAN"
print(escalation_check(angry_message, "I can help with clinic information."))
print(summary_stage(state))


In [ ]:
# Test 4: Lead qualification
state = create_initial_state()
state["stage"] = "qualification"
print(lead_qualification_stage(state))
for answer in [
    "7AM-7PM",
    "5",
    "Haircut - Rs. 100, Shave - Rs. 50, Head Massage - Rs. 100",
    "Message via WhatsApp",
    "1 hour time slots, 12 slots a day in our working hours",
]:
    add_transcript("User", answer, state)
    reply = lead_qualification_stage(state)
    add_transcript("Agent", reply, state)
    print("Agent:", reply)

print(summary_stage(state))


In [ ]:
# Test 5: Deterministic escalation checks
state = create_initial_state()
for message, response in [
    ("Can you make Botox cheaper?", "Pricing starts from the SOP amounts."),
    ("Is Botox safe if I have a heart condition?", "Please book a consultation."),
    ("I want to speak to a human", "I can help with clinic information."),
]:
    state = create_initial_state()
    print(message, "=>", escalation_check(message, response))
